Sources:
- [Delaunay Triangulation and Voronoi Diagram using OpenCV ( C++ / Python )](https://learnopencv.com/delaunay-triangulation-and-voronoi-diagram-using-opencv-c-python/)

In [1]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
from datetime import date
from uuid import uuid4
import random

# import 3rd-party modules
import cv2
import numpy as np

# import local modules
from utils.renderer.resizer import resize_with_pad, resize_with_crop
from utils.project_manager import Project

In [2]:
# name out img dirs
out_img_dir_list = ["out"]

# create project
project = Project(project_dir="assets/images/voronoi", out_img_dir_list=out_img_dir_list)

In [3]:
out_img_dir = "out"

In [67]:
img_path_list = project.get_img_path_list(img_dir="/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/images/mosaic/baba/db_baba")
# img_path_list = project.get_img_path_list(img_dir="/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/images/mosaic/snapshots/spiderman/db_40000")

In [4]:
# Check if a point is inside a rectangle
def rect_contains(rect, point) :
    if point[0] < rect[0] :
        return False
    elif point[1] < rect[1] :
        return False
    elif point[0] > rect[2] :
        return False
    elif point[1] > rect[3] :
        return False
    return True

# Draw a point
def draw_point(img, p, color ) :
    cv2.circle( img, p, 2, color, cv2.FILLED, cv2.LINE_AA, 0 )

# Draw delaunay triangles
def draw_delaunay(img, subdiv, delaunay_color ) :

    triangleList = subdiv.getTriangleList();
    size = img.shape
    r = (0, 0, size[1], size[0])

    for t in triangleList :

        pt1 = (t[0], t[1])
        pt2 = (t[2], t[3])
        pt3 = (t[4], t[5])

        if rect_contains(r, pt1) and rect_contains(r, pt2) and rect_contains(r, pt3) :

            cv2.line(img, pt1, pt2, delaunay_color, 1, cv2.LINE_AA, 0)
            cv2.line(img, pt2, pt3, delaunay_color, 1, cv2.LINE_AA, 0)
            cv2.line(img, pt3, pt1, delaunay_color, 1, cv2.LINE_AA, 0)

# Draw voronoi diagram
def draw_voronoi(img, subdiv) :

    ( facets, centers) = subdiv.getVoronoiFacetList([])

    for i in range(0,len(facets)) :
        ifacet_arr = []
        for f in facets[i] :
            ifacet_arr.append(f)

        ifacet = np.array(ifacet_arr, int)
        color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))

        cv2.fillConvexPoly(img, ifacet, color, cv2.LINE_AA, 0);
        ifacets = np.array([ifacet])
        cv2.polylines(img, ifacets, True, (0, 0, 0), 1, cv2.LINE_AA, 0)
        cv2.circle(img, (centers[i][0], centers[i][1]), 3, (0, 0, 0), cv2.FILLED, cv2.LINE_AA, 0)

# Draw voronoi diagram
def draw_voronoi_masks(ref_img_shape, subdiv, img_path_list) :

    ( facets, centers) = subdiv.getVoronoiFacetList([])

    out_img = np.zeros(ref_img_shape, dtype = np.uint8)

    error_count = 0

    for i in range(0,len(facets)) :

        mask = np.zeros(ref_img_shape[:2], dtype = np.uint8)
        canvas = np.zeros(ref_img_shape, dtype = np.uint8)

        ref_img_height, ref_img_width, ref_img_channel = ref_img_shape

        global ifacet
        
        ifacet_arr = []
        for f in facets[i] :
            ifacet_arr.append(f)

        ifacet = np.array(ifacet_arr, int)
        color = 1

        cv2.fillConvexPoly(mask, ifacet, color, cv2.LINE_AA, 0);
        ifacets = np.array([ifacet])
        # cv2.polylines(out_img, ifacets, True, (0, 0, 0), 3, cv2.LINE_AA, 0)
        # cv2.circle(mask, (centers[i][0], centers[i][1]), 3, 0, cv2.FILLED, cv2.LINE_AA, 0)

        # img = resize_with_crop(img_path = img_path_list[i], ref_img_shape = ref_img_shape)
        #  img = cv2.bitwise_and(img, img, mask=mask)

        # resize to bbox shape
        bbox = bounding_box(ifacet)
        bbox_shape = (bbox[1] - bbox[0], bbox[3] - bbox[2])

        try:
            img = resize_with_crop(img_path = img_path_list[i], ref_img_shape = (bbox_shape[0], bbox_shape[1], 3))

             # put img to bbox position, clip img if bbox falls outside of canvas
            clip_amount_min_y = 0 - bbox[0] if bbox[0] < 0 else 0
            clip_amount_min_x = 0 - bbox[2] if bbox[2] < 0 else 0
            clip_amount_max_y = ref_img_height - bbox[1] if bbox[1] > ref_img_height else 0
            clip_amount_max_x = ref_img_width - bbox[3] if bbox[3] > ref_img_width else 0
            # print(bbox[0], clip_amount_min_y, bbox[1], clip_amount_max_y)
            # print(bbox[2], clip_amount_min_x, bbox[3], clip_amount_max_x)
            # print(clip_amount_min_y, bbox_shape[0], clip_amount_max_y)
            # print(clip_amount_min_x, bbox_shape[1], clip_amount_max_x)

            canvas[
                bbox[0] + clip_amount_min_y:bbox[1] + clip_amount_max_y, 
                bbox[2] + clip_amount_min_x:bbox[3] + clip_amount_max_x, 
                :] = img[
                clip_amount_min_y:bbox_shape[0] + clip_amount_max_y, 
                clip_amount_min_x:bbox_shape[1] + clip_amount_max_x
                , :]
            
            img = cv2.bitwise_and(canvas, canvas, mask=mask)

        except:
            # img = np.ones((bbox_shape[0], bbox_shape[1], 3), dtype=np.uint8)
            img = resize_with_crop(img_path = img_path_list[i], ref_img_shape = ref_img_shape)
            img = cv2.bitwise_and(img, img, mask=mask)

            error_count += 1
            print("error_count:", error_count)
            print("bbox:", bbox)
            print("bbox_shape:", bbox_shape)

       
       
        out_img = cv2.add(out_img, img)

    return out_img


def bounding_box(points):
    x_coordinates, y_coordinates = zip(*points)

    return [min(y_coordinates), max(y_coordinates), min(x_coordinates), max(x_coordinates)]

In [70]:
# Define window names
win_delaunay = "Delaunay Triangulation"
win_voronoi = "Voronoi Diagram"

# Turn on animation while drawing triangles
animate = False

# Define colors for drawing.
delaunay_color = (255,255,255)
points_color = (0, 0, 255)

# # Read in the image.
# img = cv2.imread("image.jpg");
# img = np.ones((500, 1000, 3), dtype=np.uint8)*255
img = np.zeros((3888, 5184, 3), dtype=np.uint8)

# img = cv2.imread("assets/images/mosaic/snapshots/spiderman/to_recreate/spiderman_window.png")

# Keep a copy around
img_orig = img.copy();

# Rectangle to be used with Subdiv2D
size = img.shape
rect = (0, 0, size[1], size[0])

# Create an instance of Subdiv2D
subdiv = cv2.Subdiv2D(rect);

# Create an array of points.
points = [];

# # Read in the points from a text file
# with open("points.txt") as file :
#     for line in file :
#         x, y = line.split()
#         points.append((int(x), int(y)))

NB_POINTS = len(img_path_list)
ys = np.random.randint(0, size[0], size=(NB_POINTS,), dtype=int)
xs = np.random.randint(0, size[1], size=(NB_POINTS,), dtype=int)

points = list(zip(xs, ys))

# Insert points into subdiv
for p in points :
    subdiv.insert(p)

    # Show animation
    if animate :
        img_copy = img_orig.copy()
        # Draw delaunay triangles
        draw_delaunay( img_copy, subdiv, (255, 255, 255) );
        cv2.imshow(win_delaunay, img_copy)
        cv2.waitKey(100)

# # Draw delaunay triangles
# draw_delaunay( img, subdiv, (255, 255, 255) );

# Draw points
# for p in points :
#     draw_point(img, p, (0,0,255))

# # Allocate space for Voronoi Diagram
# img_voronoi = np.zeros(img.shape, dtype = img.dtype)

# Draw Voronoi diagram
# draw_voronoi(img_voronoi,subdiv)
out_img = draw_voronoi_masks(tuple(img.shape), subdiv, img_path_list)

# get current date
today = date.today().strftime("%Y%m%d")

# set output image path
out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{NB_POINTS}_{today}.jpg")

# save output image
cv2.imwrite(out_img_path, out_img)

# Show results
# cv2.imshow(win_delaunay,img)
# cv2.imshow(win_voronoi,img_voronoi)
# cv2.imshow(win_voronoi, out_img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
# cv2.waitKey(1)

True

## Draw delaunay, fill triangle with color

In [13]:
# Draw delaunay triangles
def draw_delaunay(
    img, subdiv, delaunay_color, line_thickness = 1, gradient_color_line=True, fill_triangle=False,
    rect_limit=True
) :

    triangleList = subdiv.getTriangleList();
    size = img.shape
    r = (0, 0, size[1], size[0])

    for t in triangleList :

        pt1 = (t[0], t[1])
        pt2 = (t[2], t[3])
        pt3 = (t[4], t[5])

        t = t.astype(int).reshape(-1, 2)

        if rect_limit and rect_contains(r, pt1) and rect_contains(r, pt2) and rect_contains(r, pt3) :

            if gradient_color_line:
                thickness_min = 1
                thickness_max = 25

                # blend image at each step
                for line_thickness in range(thickness_max, thickness_min - 1, -1 ):

                    blue_color = int(255 - line_thickness * 255 / thickness_max)
                    
                    delaunay_color = (blue_color, delaunay_color[1], delaunay_color[1])

                    cv2.line(img, pt1, pt2, delaunay_color, line_thickness, cv2.LINE_AA, 0)
                    cv2.line(img, pt2, pt3, delaunay_color, line_thickness, cv2.LINE_AA, 0)
                    cv2.line(img, pt3, pt1, delaunay_color, line_thickness, cv2.LINE_AA, 0)

            else:
                cv2.line(img, pt1, pt2, delaunay_color, line_thickness, cv2.LINE_AA, 0)
                cv2.line(img, pt2, pt3, delaunay_color, line_thickness, cv2.LINE_AA, 0)
                cv2.line(img, pt3, pt1, delaunay_color, line_thickness, cv2.LINE_AA, 0)


            if fill_triangle:
                # set random color
                color = np.random.randint(1,255, size=(3), dtype=np.uint8).tolist()
                triangle_cnt = np.array([pt1, pt2, pt3], dtype=int)

                # draw triangle contours & filled with colors
                cv2.drawContours(img, [triangle_cnt], -1, color, -1)

        else:
            if gradient_color_line:
                thickness_min = 1
                thickness_max = 25

                # blend image at each step
                for line_thickness in range(thickness_max, thickness_min - 1, -1 ):

                    blue_color = int(255 - line_thickness * 255 / thickness_max)
                    
                    delaunay_color = (blue_color, delaunay_color[1], delaunay_color[1])

                    cv2.line(img, pt1, pt2, delaunay_color, line_thickness, cv2.LINE_AA, 0)
                    cv2.line(img, pt2, pt3, delaunay_color, line_thickness, cv2.LINE_AA, 0)
                    cv2.line(img, pt3, pt1, delaunay_color, line_thickness, cv2.LINE_AA, 0)

            else:
                cv2.line(img, pt1, pt2, delaunay_color, line_thickness, cv2.LINE_AA, 0)
                cv2.line(img, pt2, pt3, delaunay_color, line_thickness, cv2.LINE_AA, 0)
                cv2.line(img, pt3, pt1, delaunay_color, line_thickness, cv2.LINE_AA, 0)


            if fill_triangle:
                # set random color
                color = np.random.randint(1,255, size=(3), dtype=np.uint8).tolist()
                triangle_cnt = np.array([pt1, pt2, pt3], dtype=int)

                # draw triangle contours & filled with colors
                cv2.drawContours(img, [triangle_cnt], -1, color, -1)

In [14]:
# Define window names
win_delaunay = "Delaunay Triangulation"
win_voronoi = "Voronoi Diagram"

# Turn on animation while drawing triangles
animate = False

# Define colors for drawing.
delaunay_color = (255,255,255)
points_color = (0, 0, 255)

# # Read in the image.
# img = cv2.imread("image.jpg");
# img = np.ones((500, 1000, 3), dtype=np.uint8)*255
img = np.zeros((3888, 5184, 3), dtype=np.uint8)

# img = cv2.imread("assets/images/mosaic/snapshots/spiderman/to_recreate/spiderman_window.png")

# Keep a copy around
img_orig = img.copy();

# Rectangle to be used with Subdiv2D
size = img.shape
rect = (0, 0, size[1], size[0])

# Create an instance of Subdiv2D
subdiv = cv2.Subdiv2D(rect);

# Create an array of points.
points = [];

# # Read in the points from a text file
# with open("points.txt") as file :
#     for line in file :
#         x, y = line.split()
#         points.append((int(x), int(y)))

NB_POINTS = 1000
ys = np.random.randint(0, size[0], size=(NB_POINTS,), dtype=int)
xs = np.random.randint(0, size[1], size=(NB_POINTS,), dtype=int)

points = list(zip(xs, ys))

# Insert points into subdiv
for p in points :
    subdiv.insert(p)

    # Show animation
    if animate :
        img_copy = img_orig.copy()
        # Draw delaunay triangles
        draw_delaunay( img_copy, subdiv, (255, 255, 255) );
        cv2.imshow(win_delaunay, img_copy)
        cv2.waitKey(100)

# Allocate space for Voronoi Diagram
img_delaunay = np.zeros_like(img)

# Draw delaunay triangles
draw_delaunay(
    img_delaunay, subdiv, delaunay_color = (255, 255, 255), 
    line_thickness = 1, gradient_color_line=False, fill_triangle=True,
    rect_limit = False
    )

# Draw points
# for p in points :
#     draw_point(img, p, (0,0,255))

# # Allocate space for Voronoi Diagram
# img_voronoi = np.zeros(img.shape, dtype = img.dtype)

# Draw Voronoi diagram
# draw_voronoi(img_voronoi,subdiv)
# out_img = draw_voronoi_masks(tuple(img.shape), subdiv, img_path_list)

# get current date
today = date.today().strftime("%Y%m%d")

# set output image path
out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{NB_POINTS}_{today}.jpg")

# save output image
cv2.imwrite(out_img_path, img_delaunay)

# Show results
# cv2.imshow(win_delaunay,img)
# cv2.imshow(win_voronoi,img_voronoi)
# cv2.imshow(win_voronoi, out_img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
# cv2.waitKey(1)

<ipython-input-13-c583cf4081a1>:67: DeprecationWarning: an integer is required (got type numpy.float32).  Implicit conversion to integers using __int__ is deprecated, and may be removed in a future version of Python.
  cv2.line(img, pt1, pt2, delaunay_color, line_thickness, cv2.LINE_AA, 0)
<ipython-input-13-c583cf4081a1>:68: DeprecationWarning: an integer is required (got type numpy.float32).  Implicit conversion to integers using __int__ is deprecated, and may be removed in a future version of Python.
  cv2.line(img, pt2, pt3, delaunay_color, line_thickness, cv2.LINE_AA, 0)
<ipython-input-13-c583cf4081a1>:69: DeprecationWarning: an integer is required (got type numpy.float32).  Implicit conversion to integers using __int__ is deprecated, and may be removed in a future version of Python.
  cv2.line(img, pt3, pt1, delaunay_color, line_thickness, cv2.LINE_AA, 0)


True

(7, 13)